This notebook runs model training + evaluation pipeline on Scheme A, random cell-level split.  

Plan:
- Justify cell type removal, plot on cell type counts
- Filter cell types
- Plots on batch, good mixing

Supplementary Section
- Run with batch variable defined
- Run on all cell types

In [ ]:
# This notebook runs the model training + evaluation pipeline for Scheme A:
# random cell-level train/test split across all transcriptomic representations.

###################SCRIPT B##########################
# --------------------
# Imports
# --------------------
import os
from pathlib import Path

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
)

# --------------------
# Set repo root and load data
# --------------------
REPO_ROOT = Path("~/scRNA-cross-donor-generalization").expanduser()  # change if needed
os.chdir(REPO_ROOT)

adata = sc.read_h5ad("data/adata_processed.h5ad")

# --------------------
# Settings
# --------------------
CELLTYPE_COL = "cell_type"
RANDOM_STATE = 42
TEST_SIZE = 0.2

results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

REPRESENTATIONS = {
    "hvg": "hvg",
    "pca": "X_pca",
    "harmony": "X_harmony",
    "scvi": "X_scVI",
}

# --------------------
# Helper functions
# --------------------
def get_representation(adata, rep_name):
    if rep_name == "hvg":
        X = adata.X
    else:
        X = adata.obsm[rep_name]

    if hasattr(X, "toarray"):
        X = X.toarray()
    else:
        X = np.asarray(X)

    return X


def run_random_split_logreg(
    adata,
    rep_name,
    celltype_col=CELLTYPE_COL,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
):
    X = get_representation(adata, rep_name)
    y = adata.obs[celltype_col].astype(str)

    counts = y.value_counts()
    keep_labels = counts[counts >= 2].index
    keep_mask = y.isin(keep_labels)

    X = X[keep_mask.to_numpy()]
    y = y[keep_mask].to_numpy()

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y,
    )

    clf = LogisticRegression(
        max_iter=5000,
        random_state=random_state,
        n_jobs=-1,
    )
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    labels = np.unique(y)
    cm = confusion_matrix(y_test, y_pred, labels=labels)

    macro_f1 = f1_score(y_test, y_pred, average="macro")
    acc = accuracy_score(y_test, y_pred)

    report = classification_report(
        y_test,
        y_pred,
        labels=labels,
        output_dict=True,
        zero_division=0,
    )

    per_class_f1 = pd.DataFrame({
        "cell_type": labels,
        "f1": [report[label]["f1-score"] for label in labels],
        "precision": [report[label]["precision"] for label in labels],
        "recall": [report[label]["recall"] for label in labels],
        "support": [report[label]["support"] for label in labels],
    })

    return {
        "model": clf,
        "y_test": y_test,
        "y_pred": y_pred,
        "macro_f1": macro_f1,
        "accuracy": acc,
        "cm": cm,
        "labels": labels,
        "per_class_f1": per_class_f1,
        "n_cells_used": len(y),
        "n_classes_used": len(labels),
    }


def save_confusion_matrix(cm, labels, out_file, title, normalize=False):
    if normalize:
        row_sums = cm.sum(axis=1, keepdims=True)
        cm_plot = np.divide(
            cm.astype(float),
            row_sums,
            out=np.zeros_like(cm, dtype=float),
            where=row_sums != 0,
        )
    else:
        cm_plot = cm

    fig, ax = plt.subplots(figsize=(12, 10))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm_plot, display_labels=labels)
    disp.plot(
        ax=ax,
        xticks_rotation=90,
        cmap="Blues",
        colorbar=True,
        values_format=None,
    )

    # remove in-cell text for readability
    if disp.text_ is not None:
        for text in disp.text_.ravel():
            if text is not None:
                text.set_visible(False)

    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(out_file, dpi=300, bbox_inches="tight")
    plt.close()


# --------------------
# Run Scheme A for all representations
# --------------------
metrics_rows = []

for rep_label, rep_key in REPRESENTATIONS.items():
    print(f"Running Scheme A for {rep_label} ({rep_key})...")

    res = run_random_split_logreg(adata, rep_name=rep_key)

    print(f"  Macro F1: {res['macro_f1']:.4f}")
    print(f"  Accuracy: {res['accuracy']:.4f}")

    # save per-class metrics
    res["per_class_f1"].to_csv(
        results_dir / f"schemeA_{rep_label}_per_class_f1.csv",
        index=False
    )

    # save full confusion matrix as csv
    cm_df = pd.DataFrame(res["cm"], index=res["labels"], columns=res["labels"])
    cm_df.to_csv(results_dir / f"schemeA_{rep_label}_confusion_matrix.csv")

    # save normalized confusion matrix plot
    save_confusion_matrix(
        res["cm"],
        res["labels"],
        results_dir / f"schemeA_{rep_label}_confusion_matrix_normalized.png",
        title=f"Scheme A Random Split - {rep_label} (row-normalized)",
        normalize=True,
    )

    # append summary row
    metrics_rows.append({
        "scheme": "random",
        "representation": rep_label,
        "macro_f1": res["macro_f1"],
        "accuracy": res["accuracy"],
        "n_cells_used": res["n_cells_used"],
        "n_classes_used": res["n_classes_used"],
    })

# --------------------
# Save overall metrics table
# --------------------
metrics_df = pd.DataFrame(metrics_rows)
metrics_df = metrics_df.sort_values("macro_f1", ascending=False).reset_index(drop=True)
metrics_df.to_csv(results_dir / "metrics.csv", index=False)

print("\nFinal Scheme A metrics:")
print(metrics_df)

Running Scheme A for hvg (hvg)...
  Macro F1: 0.4766
  Accuracy: 0.7967
Running Scheme A for pca (X_pca)...
  Macro F1: 0.5008
  Accuracy: 0.7786
Running Scheme A for harmony (X_harmony)...
  Macro F1: 0.4542
  Accuracy: 0.7533
Running Scheme A for scvi (X_scVI)...
  Macro F1: 0.5079
  Accuracy: 0.7586

Final Scheme A metrics:
   scheme representation  macro_f1  accuracy  n_cells_used  n_classes_used
0  random           scvi  0.507904  0.758636         11289              40
1  random            pca  0.500756  0.778565         11289              40
2  random            hvg  0.476643  0.796723         11289              40
3  random        harmony  0.454162  0.753322         11289              40
